In [1]:
import faiss
import numpy as np
import json

# Load embeddings and metadata
embeddings = np.load("perfume_vectors.npy")
with open("perfume_metadata.json", "r") as f:
    metadata = json.load(f)

print("Embeddings shape:", embeddings.shape)
print("Example:", metadata[0])


Embeddings shape: (24063, 1671)
Example: {'Perfume': 'accento-overdose-pride-edition', 'url': 'https://www.fragrantica.com/perfume/xerjoff/accento-overdose-pride-edition-74630.html'}


In [2]:
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

In [3]:
# define dimensionality
dim = embeddings.shape[1]

# create index
index = faiss.IndexFlatIP(dim)

# add embeddings
index.add(embeddings)

print("Total perfumes indexed:", index.ntotal)

Total perfumes indexed: 24063


In [5]:
#Testing

query = np.zeros(dim, dtype=np.float32)

note_to_idx = {note: i for i, note in enumerate(open("note_vocab.txt").read().splitlines())}

for note, confidence in {"jasmine":0.993, "sage":0.963, "amber":0.942, "ambergris":0.838}.items():
    if note in note_to_idx:
        query[note_to_idx[note]] = confidence

#normalize the query vector
query = query / np.linalg.norm(query)

#search top 5 similar perfumes
distances, indices = index.search(query.reshape(1,-1), k = 5)

for rank, (idx, score) in enumerate(zip(indices[0], distances[0]), 1):
    name = metadata[idx]["Perfume"]
    url = metadata[idx]["url"]
    print(f"{rank}. {name} ({score:.3f}) — {url}")

1. amalia-primavera (0.677) — https://www.fragrantica.com/perfume/fueguia-1833/amalia-primavera-18239.html
2. soft-and-young (0.669) — https://www.fragrantica.com/perfume/verset-parfums/soft-and-young-59213.html
3. white-gardenia (0.669) — https://www.fragrantica.com/perfume/zara/white-gardenia-68681.html
4. white-luminous-gold (0.669) — https://www.fragrantica.com/perfume/michael-kors/white-luminous-gold-31343.html
5. mysore-incenza (0.623) — https://www.fragrantica.com/perfume/areej-le-dore/mysore-incenza-76652.html


In [6]:
faiss.write_index(index, "perfume_index.faiss")

In [1]:
#creating the faiss again

import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import os

/Users/hirunweththewa/Desktop/ScentAI/.env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
with open("perfumes.json", "r", encoding="utf-8") as f:
    perfumes = json.load(f)

documents = []
faiss_id_mapping = []

for p in perfumes:
    top = ", ".join(p["top"]) if p["top"] else "none"
    middle = ", ".join(p["middle"]) if p["middle"] else "none"
    base = ", ".join(p["base"]) if p["base"] else "none"
    accords = ", ".join(p["accords"]) if p["accords"] else "various"
    
    # Handle missing brands
    brand_text = f" by {p['brand']}" if p['brand'] != "Unknown" else ""
    
    # Build a natural, descriptive sentence
    doc = (
        f"This is a {p['gender']} fragrance{brand_text} called {p['perfume']}. "
        f"The main accords are {accords}. "
        f"It opens with top notes of {top}, transitions to heart notes of {middle}, "
        f"and dries down to base notes of {base}."
    )
    
    documents.append(doc)
    faiss_id_mapping.append(p["id"])

model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Embedding {len(documents)} perfumes...")
embeddings = model.encode(documents, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.array(embeddings, dtype=np.float32)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

os.makedirs("./Data", exist_ok=True)
faiss.write_index(index, "perfume_index.bin")

with open("./Data/faiss_mapping.json", "w") as f:
    json.dump(faiss_id_mapping, f)

print("FAISS Index successfully built and saved to perfume_index.bin!")

Embedding 24063 perfumes...


Batches: 100%|██████████| 752/752 [00:53<00:00, 14.00it/s]

FAISS Index successfully built and saved to perfume_index.bin!
